# Text Mining: Klasifikasi Berita Finance vs Sport

Notebook ini menyajikan pipeline lengkap text mining untuk klasifikasi berita Detikcom kategori Finance dan Sport.

**Alur Kerja:**
```
[Dataset 200 Berita (100 Finance + 100 Sport)]
       |
       v
[1. Load & Preprocessing] --> Cleaning, Tokenization, Remove Stopwords
       |
       v
[2. Feature Extraction (TF-IDF)] --> Vektorisasi teks menjadi matriks numerik
       |
       v
[3. Dimensionality Reduction (PCA)] --> Reduksi dari 2000 fitur ke 50 komponen
       |
       v
[4. Train-Test Split] --> 80% Training, 20% Testing
       |
       v
[5. Machine Learning Models] --> Naive Bayes, SVM, Logistic Regression
       |
       v
[6. Evaluation] --> Accuracy, Precision, Recall, F1-Score, Confusion Matrix
```

**Mahasiswa:** Wahyu Pratama | **NPM:** 230411100058

## 1. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter

# Scikit-learn untuk machine learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Konfigurasi visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_palette("husl")

print("Library berhasil dimuat")

## 2. Load Dataset

Memuat dataset berita finance dan sport dari file CSV.

In [ ]:
# Load dataset
df_finance = pd.read_csv('detik_finance_100.csv')
df_sport = pd.read_csv('detik_sport_100.csv')

# Tambahkan label
df_finance['label'] = 1  # Label 1 untuk Finance
df_sport['label'] = 2    # Label 2 untuk Sport

# Gabungkan dataset
df = pd.concat([df_finance, df_sport], ignore_index=True)

print(f"Total dokumen: {len(df)}")
print(f"\nDistribusi kelas:")
print(df['label'].value_counts())

# Tampilkan contoh data
print("\nContoh data:")
df[['judul', 'label']].head()

## 3. Text Preprocessing

Tahapan preprocessing teks:
- Lowercase: mengubah semua teks menjadi huruf kecil
- Remove special characters: menghapus karakter khusus dan angka
- Remove extra spaces: menghapus spasi berlebih

In [ ]:
def preprocess_text(text):
    """Fungsi preprocessing teks sederhana"""
    if pd.isna(text):
        return ""
    # Lowercase
    text = str(text).lower()
    # Hapus karakter khusus, hanya ambil huruf dan spasi
    text = re.sub(r'[^a-z\s]', ' ', text)
    # Hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Terapkan preprocessing
df['teks_bersih'] = df['isi_berita'].apply(preprocess_text)

print("Contoh hasil preprocessing:")
print(f"\nOriginal:\n{df.iloc[0]['isi_berita'][:200]}...")
print(f"\nSetelah preprocessing:\n{df.iloc[0]['teks_bersih'][:200]}...")

# Statistik kata
all_words = ' '.join(df['teks_bersih']).split()
unique_words = set(all_words)
print(f"\nTotal kata: {len(all_words):,}")
print(f"Kata unik: {len(unique_words):,}")

## 4. Feature Extraction dengan TF-IDF

TF-IDF (Term Frequency - Inverse Document Frequency) mengubah teks menjadi representasi numerik.
- TF: frekuensi kata dalam dokumen
- IDF: mengukur seberapa penting kata tersebut di seluruh corpus
- Formula: TF-IDF = TF × log(N / df)

Parameter:
- max_features=2000: ambil 2000 kata terpenting
- min_df=2: kata harus muncul minimal di 2 dokumen
- max_df=0.95: buang kata yang muncul di >95% dokumen

In [ ]:
# Inisialisasi TF-IDF Vectorizer
vectorizer = TfidfVectorizer(
    max_features=2000,  # Ambil 2000 fitur terbaik
    min_df=2,           # Kata harus muncul minimal di 2 dokumen
    max_df=0.95,        # Buang kata yang terlalu umum (>95% dokumen)
    tokenizer=lambda x: x.split()  # Tokenizer sederhana
)

# Transform teks menjadi matriks TF-IDF
X_tfidf = vectorizer.fit_transform(df['teks_bersih'])
feature_names = vectorizer.get_feature_names_out()

print(f"Matriks TF-IDF: {X_tfidf.shape}")
print(f"Jumlah fitur (kata): {len(feature_names)}")
print(f"\n10 kata pertama: {list(feature_names[:10])}")

# Konversi ke DataFrame untuk visualisasi
tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=feature_names
)
tfidf_df['label'] = df['label'].values

print(f"\nContoh nilai TF-IDF (5 dokumen, 10 kata):")
tfidf_df.iloc[:5, :10]

### 4.1 Analisis Kata Paling Penting per Kategori

In [ ]:
# Hitung rata-rata TF-IDF per kategori
finance_tfidf = tfidf_df[df['label'] == 1].iloc[:, :-1].mean().sort_values(ascending=False)
sport_tfidf = tfidf_df[df['label'] == 2].iloc[:, :-1].mean().sort_values(ascending=False)

print("Top 15 kata paling penting untuk FINANCE:")
print(finance_tfidf.head(15))

print("\nTop 15 kata paling penting untuk SPORT:")
print(sport_tfidf.head(15))

# Visualisasi
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

finance_tfidf.head(15).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 15 Kata - Finance', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Rata-rata TF-IDF')
axes[0].invert_yaxis()

sport_tfidf.head(15).plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Top 15 Kata - Sport', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Rata-rata TF-IDF')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 5. Dimensionality Reduction dengan PCA

PCA (Principal Component Analysis) mereduksi dimensi data dari 2000 fitur menjadi 50 komponen utama.
- Tujuan: mengurangi kompleksitas, mempercepat training, menghilangkan noise
- PC1 (Principal Component 1): komponen dengan variance terbesar
- Variance explained: persentase informasi yang dipertahankan

In [ ]:
# PCA dengan 50 komponen
n_components = 50
pca = PCA(n_components=n_components, random_state=42)
X_pca = pca.fit_transform(X_tfidf.toarray())

# Hitung variance explained
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

print(f"Shape setelah PCA: {X_pca.shape}")
print(f"Variance explained oleh 10 komponen pertama: {cumulative_variance[9]:.2%}")
print(f"Variance explained oleh semua {n_components} komponen: {cumulative_variance[-1]:.2%}")

# Visualisasi variance explained
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Individual variance
axes[0].bar(range(1, n_components+1), explained_variance)
axes[0].set_title('Variance Explained per Komponen', fontweight='bold')
axes[0].set_xlabel('Komponen PCA')
axes[0].set_ylabel('Variance Explained')
axes[0].grid(alpha=0.3)

# Cumulative variance
axes[1].plot(range(1, n_components+1), cumulative_variance, marker='o')
axes[1].set_title('Cumulative Variance Explained', fontweight='bold')
axes[1].set_xlabel('Jumlah Komponen')
axes[1].set_ylabel('Cumulative Variance')
axes[1].axhline(y=0.8, color='r', linestyle='--', label='80% threshold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 5.1 Visualisasi 2D PCA (PC1 vs PC2)

In [ ]:
# Scatter plot PC1 vs PC2
plt.figure(figsize=(10, 7))

finance_idx = df['label'] == 1
sport_idx = df['label'] == 2

plt.scatter(X_pca[finance_idx, 0], X_pca[finance_idx, 1], 
           c='steelblue', label='Finance', alpha=0.6, s=50, edgecolors='k')
plt.scatter(X_pca[sport_idx, 0], X_pca[sport_idx, 1], 
           c='coral', label='Sport', alpha=0.6, s=50, edgecolors='k')

plt.xlabel(f'PC1 ({explained_variance[0]:.1%} variance)', fontsize=12)
plt.ylabel(f'PC2 ({explained_variance[1]:.1%} variance)', fontsize=12)
plt.title('Visualisasi PCA: Finance vs Sport', fontsize=14, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Train-Test Split

Membagi dataset menjadi:
- Training set (80%): untuk melatih model
- Testing set (20%): untuk evaluasi performa model

In [ ]:
# Ambil label
y = df['label'].values

# Split data (80% train, 20% test)
X_train_tfidf, X_test_tfidf, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

X_train_pca, X_test_pca, _, _ = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)

print("Split Dataset:")
print(f"Training set: {X_train_tfidf.shape[0]} dokumen")
print(f"Testing set: {X_test_tfidf.shape[0]} dokumen")
print(f"\nDistribusi label training:")
print(pd.Series(y_train).value_counts())
print(f"\nDistribusi label testing:")
print(pd.Series(y_test).value_counts())

## 7. Machine Learning Models

Melatih dan mengevaluasi 3 algoritma klasifikasi:
1. **Multinomial Naive Bayes**: probabilistic classifier, cocok untuk text classification
2. **Linear SVM**: mencari hyperplane terbaik untuk memisahkan kelas
3. **Logistic Regression**: regresi untuk klasifikasi binary/multiclass

### 7.1 Multinomial Naive Bayes

In [ ]:
# Training Naive Bayes dengan TF-IDF
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

# Prediksi
y_pred_nb = nb_model.predict(X_test_tfidf)

# Evaluasi
acc_nb = accuracy_score(y_test, y_pred_nb)
print("=" * 60)
print("NAIVE BAYES - TF-IDF")
print("=" * 60)
print(f"Accuracy: {acc_nb:.4f} ({acc_nb*100:.2f}%)")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb, target_names=['Finance', 'Sport']))

# Confusion Matrix
cm_nb = confusion_matrix(y_test, y_pred_nb)
plt.figure(figsize=(7, 6))
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Blues', 
           xticklabels=['Finance', 'Sport'], 
           yticklabels=['Finance', 'Sport'])
plt.title('Confusion Matrix - Naive Bayes', fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

### 7.2 Linear SVM

In [ ]:
# Training SVM dengan TF-IDF
svm_model = LinearSVC(random_state=42, max_iter=2000)
svm_model.fit(X_train_tfidf, y_train)

# Prediksi
y_pred_svm = svm_model.predict(X_test_tfidf)

# Evaluasi
acc_svm = accuracy_score(y_test, y_pred_svm)
print("=" * 60)
print("LINEAR SVM - TF-IDF")
print("=" * 60)
print(f"Accuracy: {acc_svm:.4f} ({acc_svm*100:.2f}%)")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm, target_names=['Finance', 'Sport']))

# Confusion Matrix
cm_svm = confusion_matrix(y_test, y_pred_svm)
plt.figure(figsize=(7, 6))
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Greens', 
           xticklabels=['Finance', 'Sport'], 
           yticklabels=['Finance', 'Sport'])
plt.title('Confusion Matrix - Linear SVM', fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

### 7.3 Logistic Regression

In [ ]:
# Training Logistic Regression dengan TF-IDF
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)

# Prediksi
y_pred_lr = lr_model.predict(X_test_tfidf)

# Evaluasi
acc_lr = accuracy_score(y_test, y_pred_lr)
print("=" * 60)
print("LOGISTIC REGRESSION - TF-IDF")
print("=" * 60)
print(f"Accuracy: {acc_lr:.4f} ({acc_lr*100:.2f}%)")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['Finance', 'Sport']))

# Confusion Matrix
cm_lr = confusion_matrix(y_test, y_pred_lr)
plt.figure(figsize=(7, 6))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Oranges', 
           xticklabels=['Finance', 'Sport'], 
           yticklabels=['Finance', 'Sport'])
plt.title('Confusion Matrix - Logistic Regression', fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

### 7.4 Model dengan PCA (50 komponen)

In [ ]:
# Training dengan data PCA
nb_pca = MultinomialNB()
svm_pca = LinearSVC(random_state=42, max_iter=2000)
lr_pca = LogisticRegression(random_state=42, max_iter=1000)

# Konversi PCA ke non-negative untuk Naive Bayes
X_train_pca_pos = X_train_pca - X_train_pca.min() + 1
X_test_pca_pos = X_test_pca - X_test_pca.min() + 1

nb_pca.fit(X_train_pca_pos, y_train)
svm_pca.fit(X_train_pca, y_train)
lr_pca.fit(X_train_pca, y_train)

# Prediksi
y_pred_nb_pca = nb_pca.predict(X_test_pca_pos)
y_pred_svm_pca = svm_pca.predict(X_test_pca)
y_pred_lr_pca = lr_pca.predict(X_test_pca)

# Accuracy
acc_nb_pca = accuracy_score(y_test, y_pred_nb_pca)
acc_svm_pca = accuracy_score(y_test, y_pred_svm_pca)
acc_lr_pca = accuracy_score(y_test, y_pred_lr_pca)

print("=" * 60)
print("MODEL DENGAN PCA (50 komponen)")
print("=" * 60)
print(f"Naive Bayes + PCA    : {acc_nb_pca:.4f} ({acc_nb_pca*100:.2f}%)")
print(f"Linear SVM + PCA     : {acc_svm_pca:.4f} ({acc_svm_pca*100:.2f}%)")
print(f"Logistic Reg + PCA   : {acc_lr_pca:.4f} ({acc_lr_pca*100:.2f}%)")

## 8. Perbandingan Semua Model

In [ ]:
# Ringkasan hasil
results = pd.DataFrame({
    'Model': [
        'Naive Bayes (TF-IDF)',
        'Linear SVM (TF-IDF)',
        'Logistic Reg (TF-IDF)',
        'Naive Bayes (PCA)',
        'Linear SVM (PCA)',
        'Logistic Reg (PCA)'
    ],
    'Accuracy': [
        acc_nb,
        acc_svm,
        acc_lr,
        acc_nb_pca,
        acc_svm_pca,
        acc_lr_pca
    ]
})

results = results.sort_values('Accuracy', ascending=False).reset_index(drop=True)

print("\n" + "=" * 60)
print("RINGKASAN PERBANDINGAN MODEL")
print("=" * 60)
print(results.to_string(index=False))

# Visualisasi
plt.figure(figsize=(12, 6))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
bars = plt.barh(results['Model'], results['Accuracy'], color=colors)

# Tambahkan nilai di ujung bar
for i, (bar, acc) in enumerate(zip(bars, results['Accuracy'])):
    plt.text(acc + 0.005, i, f'{acc:.4f}', va='center', fontweight='bold')

plt.xlabel('Accuracy', fontsize=12)
plt.title('Perbandingan Accuracy Semua Model', fontsize=14, fontweight='bold')
plt.xlim(0, 1.1)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# Model terbaik
best_model = results.iloc[0]
print(f"\nModel Terbaik: {best_model['Model']}")
print(f"Accuracy: {best_model['Accuracy']:.4f} ({best_model['Accuracy']*100:.2f}%)")

## 9. Kesimpulan

Pipeline text mining ini berhasil mengklasifikasikan berita Finance dan Sport dengan tahapan:

1. **Preprocessing**: Cleaning teks, lowercase, remove special characters
2. **Feature Extraction**: TF-IDF menghasilkan 2000 fitur dari teks
3. **Dimensionality Reduction**: PCA mereduksi menjadi 50 komponen (mempertahankan ~53% variance)
4. **Classification**: Diuji dengan 3 algoritma (Naive Bayes, SVM, Logistic Regression)
5. **Evaluation**: Model dievaluasi menggunakan accuracy, precision, recall, dan F1-score

**Hasil:**
- Model dengan TF-IDF umumnya lebih akurat daripada PCA
- PCA berguna untuk visualisasi dan mempercepat training
- Dataset Finance dan Sport dapat dibedakan dengan baik

**Penulis:** Wahyu Pratama | **NPM:** 230411100058